### CA

In [2]:
import os.path as osp
import pandas as pd
import json
import shapely
import os

data_path = "/data/home/dswang/new_disk/dswang/code/DPO4POI/new_code/data/CA"
raw_checkins = pd.read_csv(osp.join(data_path, 'loc-gowalla_totalCheckins.txt'), sep='\t', header=None)
raw_checkins.columns = ['userid', 'datetime', 'checkins_lat', 'checkins_lng', 'id']
subset1 = pd.read_csv(osp.join(data_path, 'gowalla_spots_subset1.csv'))
raw_checkins_subset1 = raw_checkins.merge(subset1, on='id')

with open(osp.join(data_path, 'us_state_polygon_json.json'), 'r') as f:
    us_state_polygon = json.load(f)

for i in us_state_polygon['features']:
    if i['properties']['name'].lower() == 'california':
        california = shapely.polygons(i['geometry']['coordinates'][0])
    if i['properties']['name'].lower() == 'nevada':
        nevada = shapely.polygons(i['geometry']['coordinates'][0])

# check if the checkin took place in California or Nevada
raw_checkins_subset1['is_ca'] = raw_checkins_subset1.apply(
    lambda x: nevada.intersects(
        shapely.geometry.Point(x['checkins_lng'], x['checkins_lat'])) or california.intersects(
        shapely.geometry.Point(x['checkins_lng'], x['checkins_lat'])), axis=1
)
raw_checkins_subset1 = raw_checkins_subset1[raw_checkins_subset1['is_ca']]

df = raw_checkins_subset1[['userid', 'id', 'spot_categories', 'checkins_lat', 'checkins_lng', 'datetime']]
df.columns = ['UId', 'PId', 'CategoryId', 'Latitude', 'Longitude', 'UTCTime']
df['TimeOffset'] = -420


df.to_csv(osp.join(data_path, 'raw.csv'), index=False)

/data/home/dswang/new_disk/tmp/ipykernel_4007594/3163557372.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['TimeOffset'] = -480


In [ ]:
import pandas as pd
import ast

# 读取 CSV
df = pd.read_csv("CA/raw.csv")

# 提取 CategoryId 中的 name
def extract_name(x):
    try:
        obj = ast.literal_eval(x)
        return obj[0]["name"] if isinstance(obj, list) and len(obj) > 0 else None
    except:
        return None

df["Category"] = df["CategoryId"].apply(extract_name)

# 解析 UTC 时间（带时区）
utc_dt = pd.to_datetime(df["UTCTime"], utc=True)

# 标准化 UTCTime 字符串
df["UTCTime"] = utc_dt.dt.strftime("%Y-%m-%d %H:%M:%S")

# 确保 TimeOffset 为整数
df["TimeOffset"] = df["TimeOffset"].astype(int)

# 计算本地时间
time_offsets = pd.to_timedelta(df["TimeOffset"], unit="m")
local_dt = utc_dt + time_offsets
df["Time"] = local_dt.dt.strftime("%Y-%m-%d %H:%M:%S")

# 基于本地时间计算 Weekday
df["Weekday"] = local_dt.dt.day_name()

# 调整列顺序
df = df[
    ["UId", "PId", "Category", "Latitude", "Longitude", "Time", "Weekday"]
]

df.to_csv("CA/CA.csv", index=False)

### NYC & TKY

In [ ]:
import pandas as pd

def process_data(data_file):
    # 读取 txt（tab 分隔）
    df = pd.read_csv(data_file,sep="\t",header=None,engine="python")

    # 只保留需要的列并重命名
    df = df[[0, 1, 3, 4, 5, 6, 7]]
    df.columns = ["UId", "PId", "Category", "Latitude", "Longitude", "TimeOffset", "UTCTime"]

    df["TimeOffset"] = pd.to_numeric(df["TimeOffset"], errors="coerce")

    utc_dt = pd.to_datetime(df["UTCTime"], utc=True)

    df["UTCTime"] = utc_dt.dt.strftime("%Y-%m-%d %H:%M:%S")

    time_offsets = pd.to_timedelta(df["TimeOffset"], unit="m")
    local_dt = utc_dt + time_offsets
    df["Time"] = local_dt.dt.strftime("%Y-%m-%d %H:%M:%S")

    df["Weekday"] = local_dt.dt.day_name()

    df = df[
        ["UId", "PId", "Category", "Latitude", "Longitude", "Time", "Weekday"]
    ]

    df.to_csv(f"{data_file.split('.')[0]}.csv", index=False)


process_data("NYC/NYC.txt")
process_data("TKY/TKY.txt")

/data/home/dswang/new_disk/tmp/ipykernel_4007594/1037991073.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  utc_dt = pd.to_datetime(df["UTCTime"], utc=True)
/data/home/dswang/new_disk/tmp/ipykernel_4007594/1037991073.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  utc_dt = pd.to_datetime(df["UTCTime"], utc=True)
